In [31]:
import os
import pandas as pd
import numpy as np
import psycopg2
from psycopg2.extras import execute_values

In [32]:
# setups

connection = {
    "host":     "localhost",
    "port":     5432,
    "dbname":   "dreamhomes",
    "user":     "postgres",
    "password": "123",
}

dir = ".../dreamhomes_raw_csv_v2"

In [33]:
# define helper functions

def connect():
    return psycopg2.connect(**connection)

def load_csv(filename):
    path = os.path.join(dir, filename)
    df = pd.read_csv(path, dtype=str)
    df = df.replace({
        np.nan: None,
        "nan": None,
        "NaN": None,
        "None": None,
        "": None,
        "NULL": None,
        "null": None
    })
    return df


def to_bool(val):
    if val is None:
        return None
    return str(val).strip().lower() in ("true", "1", "yes")


def to_int(val):
    if val is None:
        return None
    try:
        return int(float(val))
    except (ValueError, TypeError):
        return None


def to_float(val):
    if val is None:
        return None
    try:
        return float(val)
    except (ValueError, TypeError):
        return None


def clean_date(val):
    if val is None:
        return None
    if str(val).strip() in ("", "None", "nan", "NaN", "NULL", "null"):
        return None
    return val

### **Loading Functions**

In [34]:
# table Offices 

def load_offices(conn, df):
    rows = []
    for i, r in enumerate(df.itertuples(index=False), start=1):
        rows.append((
            i,
            r.office_name,
            r.office_state,
            r.office_city,
            r.office_street_address,
            r.office_zip_code,
            r.office_phone_number,
            to_float(r.office_rent),
        ))

    sql = """
        INSERT INTO Offices
            (office_id, office_name, state, city, street_address, zip_code, phone_number, office_rent)
        VALUES %s
        ON CONFLICT DO NOTHING
    """

    with conn.cursor() as cur:
        execute_values(cur, sql, rows)

    conn.commit()
    return {r[1]: r[0] for r in rows}

In [35]:
# table Neighbourhoods

def load_neighbourhoods(conn, df):
    neigh_df = df[
        [
            "neighbourhood_name",
            "school_zone",
            "is_near_public_transit",
            "is_pet_friendly",
            "has_children_playground"
        ]
    ].drop_duplicates(subset=["neighbourhood_name"]).reset_index(drop=True)

    rows = []
    name_to_id = {}

    for i, r in enumerate(neigh_df.itertuples(index=False), start=1):
        rows.append((
            i,
            r.neighbourhood_name,
            r.school_zone,
            to_bool(r.is_near_public_transit),
            to_bool(r.is_pet_friendly),
            to_bool(r.has_children_playground),
        ))
        name_to_id[r.neighbourhood_name] = i

    sql = """
        INSERT INTO Neighbourhoods
            (neighbourhood_id, neighbourhood_name, school_zone,
             is_near_public_transit, is_pet_friendly, has_children_playground)
        VALUES %s
        ON CONFLICT DO NOTHING
    """

    with conn.cursor() as cur:
        execute_values(cur, sql, rows)

    conn.commit()
    return name_to_id

In [36]:
# table Clients

def load_clients(conn, df):
    rows = []
    email_to_id = {}

    for i, r in enumerate(df.itertuples(index=False), start=1):
        rows.append((
            i,
            r.first_name,
            r.last_name,
            r.email,
            r.phone,
            clean_date(r.registration_date),
        ))
        email_to_id[r.email] = i

    sql = """
        INSERT INTO Clients
            (client_id, first_name, last_name, email, phone, registration_date)
        VALUES %s
        ON CONFLICT DO NOTHING
    """

    with conn.cursor() as cur:
        execute_values(cur, sql, rows)

    conn.commit()
    return email_to_id

In [37]:
# table Agents

def load_agents(conn, df, office_name_to_id):
    rows = []
    email_to_id = {}

    for i, r in enumerate(df.itertuples(index=False), start=1):
        office_id = office_name_to_id.get(r.office_name)

        rows.append((
            i,
            office_id,
            r.agent_first_name,
            r.agent_last_name,
            r.agent_email,
            r.agent_phone,
            clean_date(r.agent_hire_date),
            r.agent_license_number,
            to_float(r.agent_base_salary),
        ))

        email_to_id[r.agent_email] = i

    sql = """
        INSERT INTO Agents
            (agent_id, office_id, first_name, last_name, email, phone,
             hire_date, license_number, base_salary)
        VALUES %s
        ON CONFLICT DO NOTHING
    """

    with conn.cursor() as cur:
        execute_values(cur, sql, rows)

    conn.commit()
    return email_to_id

In [38]:
# table Properties

def load_properties(conn, df, neigh_name_to_id):
    rows = []
    address_to_id = {}

    for i, r in enumerate(df.itertuples(index=False), start=1):
        neigh_id = neigh_name_to_id.get(r.neighbourhood_name)

        rows.append((
            i,
            r.property_type,
            r.property_street_address,
            r.property_city,
            r.property_state,
            neigh_id,
            to_int(r.bedrooms),
            to_int(r.bathrooms),
            to_int(r.square_feet),
            to_int(r.year_built),
            "Available", # hard-code now as a placeholder. will be edited later in the script
        ))

        address_to_id[r.property_full_address] = i

    sql = """
        INSERT INTO Properties
            (property_id, property_type, street_address, city, state,
             neighbourhood_id, bedrooms, bathrooms, square_feet, year_built, current_status)
        VALUES %s
        ON CONFLICT DO NOTHING
    """

    with conn.cursor() as cur:
        execute_values(cur, sql, rows)

    conn.commit()
    return address_to_id

In [39]:
# tables Appointments and AppointmentInteractions

def load_appointments(conn, df_appts, office_name_to_id, prop_addr_to_id,
                      agent_email_to_id, client_email_to_id):
    appt_rows = []
    interaction_rows = []
    appt_id = 1
    interaction_id = 1

    for r in df_appts.itertuples(index=False):
        office_id = office_name_to_id.get(r.office_name)
        prop_id = prop_addr_to_id.get(r.property_full_address)
        if office_id is None or prop_id is None:
            continue
        appt_rows.append((
            appt_id,
            office_id,
            prop_id,
            r.appointment_type,
            r.appointment_status,
            r.appointment_notes,
        ))

        agent_emails = [e.strip() for e in str(r.agent_emails).split(";") if e.strip()]
        client_emails = [e.strip() for e in str(r.client_emails).split(";") if e.strip()]

        for ae in agent_emails:
            a_id = agent_email_to_id.get(ae)
            if a_id is None:
                continue
            for ce in client_emails:
                c_id = client_email_to_id.get(ce)
                if c_id is None:
                    continue
                interaction_rows.append((
                    interaction_id,
                    appt_id,
                    a_id,
                    c_id,
                    clean_date(r.interaction_datetime),
                ))
                interaction_id += 1
        appt_id += 1

    with conn.cursor() as cur:
        if appt_rows:
            execute_values(cur, """
                INSERT INTO Appointments
                    (appointment_id, office_id, property_id, appointment_type, status, notes)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, appt_rows)

        if interaction_rows:
            execute_values(cur, """
                INSERT INTO AppointmentInteractions
                    (interaction_id, appointment_id, agent_id, client_id, interaction_datetime)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, interaction_rows)

    conn.commit()

In [40]:
# table OpenHouses

def load_openhouses(conn, df_oh, agent_email_to_id, prop_addr_to_id):
    rows = []

    for i, r in enumerate(df_oh.itertuples(index=False), start=1):
        agent_id = agent_email_to_id.get(r.hosting_agent_email)
        prop_id = prop_addr_to_id.get(r.property_full_address)
        if agent_id is None or prop_id is None:
            continue
        rows.append((
            i,
            agent_id,
            prop_id,
            clean_date(r.start_time),
            clean_date(r.end_time),
            to_float(r.cost),
            r.notes,
        ))

    with conn.cursor() as cur:
        if rows:
            execute_values(cur, """
                INSERT INTO OpenHouses
                    (open_house_id, hosting_agent_id, property_id,
                     start_time, end_time, cost, notes)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, rows)

    conn.commit()

In [41]:
# table Listings, Transcations, TransactionAgents, TransactionParties, Commissions and Expenses

def load_transactions(conn, df_tx, prop_addr_to_id,
                      agent_email_to_id, client_email_to_id):
    tx_groups = {}

    for r in df_tx.itertuples(index=False):
        ref = r.transaction_reference
        if ref not in tx_groups:
            tx_groups[ref] = {
                "meta": r,
                "participants": []
            }
        tx_groups[ref]["participants"].append(r)

    listing_rows = []
    tx_rows = []
    tx_agent_rows = []
    tx_party_rows = []
    commission_rows = []
    expense_rows = []

    status_map = {
        "sold":             ("Closed",    "Completed"),
        "pending sale":     ("Pending",   "Pending"),
        "cancelled sale":   ("Cancelled", "Cancelled"),
        "expired":          ("Expired",   "Cancelled"),
        "rented":           ("Closed",    "Completed"),
        "pending rental":   ("Pending",   "Pending"),
        "cancelled rental": ("Cancelled", "Cancelled"),
    }

    listing_id = 1
    tx_id = 1
    expense_id = 1

    for ref, group in tx_groups.items():
        meta = group["meta"]
        participants = group["participants"]
        prop_id = prop_addr_to_id.get(meta.property_full_address)

        if prop_id is None:
            continue
        listing_agent_id = None

        for p in participants:
            if p.participant_type == "Agent" and p.participant_role in ("Seller Agent", "Landlord Agent"):
                listing_agent_id = agent_email_to_id.get(p.participant_email)
                break

        if listing_agent_id is None:
            for p in participants:
                if p.participant_type == "Agent":
                    listing_agent_id = agent_email_to_id.get(p.participant_email)
                    break

        raw_status = str(meta.status).strip().lower() if meta.status else "expired"
        listing_status, tx_status = status_map.get(raw_status, ("Expired", "Cancelled"))

        listing_rows.append((
            listing_id,
            prop_id,
            listing_agent_id,
            meta.transaction_type,
            clean_date(meta.transaction_start_date),
            clean_date(meta.closing_date),
            listing_status,
        ))

        tx_rows.append((
            tx_id,
            listing_id,
            meta.transaction_type,
            tx_status,
            clean_date(meta.date_of_transaction),
            clean_date(meta.closing_date),
            to_float(meta.final_price),
            ref,
        ))

        seen_agent_roles = set()
        seen_client_roles = set()

        for p in participants:
            if p.participant_type == "Agent":
                a_id = agent_email_to_id.get(p.participant_email)
                if a_id and (a_id, p.participant_role) not in seen_agent_roles:
                    seen_agent_roles.add((a_id, p.participant_role))
                    tx_agent_rows.append((tx_id, a_id, p.participant_role))
                    comm = to_float(p.commission_amount)
                    if comm is not None:
                        commission_rows.append((a_id, tx_id, comm))
            elif p.participant_type == "Client":
                c_id = client_email_to_id.get(p.participant_email)
                if c_id and (c_id, p.participant_role) not in seen_client_roles:
                    seen_client_roles.add((c_id, p.participant_role))
                    tx_party_rows.append((tx_id, c_id, p.participant_role))

        if tx_status == "Completed":
            legal = to_float(meta.legal_fees)
            other = to_float(meta.other_expenses)
            if legal is not None or other is not None:
                expense_rows.append((
                    expense_id,
                    tx_id,
                    0 if legal is None else legal,
                    0 if other is None else other,
                ))
                expense_id += 1
        listing_id += 1
        tx_id += 1

    with conn.cursor() as cur:
        if listing_rows:
            execute_values(cur, """
                INSERT INTO Listings
                    (listing_id, property_id, agent_id, listing_type,
                     start_date, end_date, listing_status)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, listing_rows)

        if tx_rows:
            execute_values(cur, """
                INSERT INTO Transactions
                    (transaction_id, listing_id, transaction_type, transaction_status,
                     contract_date, closing_date, final_price, notes)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, tx_rows)

        if tx_agent_rows:
            execute_values(cur, """
                INSERT INTO TransactionAgents
                    (transaction_id, agent_id, agent_role)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, tx_agent_rows)

        if tx_party_rows:
            execute_values(cur, """
                INSERT INTO TransactionParties
                    (transaction_id, client_id, role_type)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, tx_party_rows)

        if commission_rows:
            execute_values(cur, """
                INSERT INTO Commissions
                    (agent_id, transaction_id, commission_amount)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, commission_rows)

        if expense_rows:
            execute_values(cur, """
                INSERT INTO Expenses
                    (expense_id, transaction_id, legal_fees, other_expenses)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, expense_rows)

    conn.commit()

In [42]:
# update property status accordingly

def update_property_status(conn):
    sql = """
        WITH latest_tx AS (
            SELECT DISTINCT ON (l.property_id)
                l.property_id,
                l.listing_status,
                t.transaction_type,
                t.contract_date,
                t.transaction_id
            FROM Listings l
            JOIN Transactions t
                ON l.listing_id = t.listing_id
            ORDER BY l.property_id, t.contract_date DESC, t.transaction_id DESC
        )
        UPDATE Properties p
        SET current_status =
            CASE
                WHEN lt.listing_status = 'Closed' AND lt.transaction_type = 'Sale' THEN 'Sold'
                WHEN lt.listing_status = 'Closed' AND lt.transaction_type = 'Rental' THEN 'Rented'
                WHEN lt.listing_status = 'Pending' THEN 'Under Contract'
                WHEN lt.listing_status IN ('Cancelled', 'Expired') THEN 'Off Market'
                ELSE 'Available'
            END
        FROM latest_tx lt
        WHERE p.property_id = lt.property_id
    """

    with conn.cursor() as cur:
        cur.execute(sql)
    conn.commit()

In [43]:
# table PropertyOwners

def load_property_owners(conn, df_tx, prop_addr_to_id, client_email_to_id):

    tx_groups = {}
    for r in df_tx.itertuples(index=False):
        ref = r.transaction_reference
        if ref not in tx_groups:
            tx_groups[ref] = {
                "meta": r,
                "participants": []
            }
        tx_groups[ref]["participants"].append(r)

    owner_rows = []
    property_owner_id = 1
    seller_roles = {"Seller", "Landlord"}
    buyer_roles = {"Buyer"}
    completed_sale_statuses = {"sold"}
    seen_ownership_records = set()

    for ref, group in tx_groups.items():
        meta = group["meta"]
        participants = group["participants"]
        prop_id = prop_addr_to_id.get(meta.property_full_address)
        if prop_id is None:
            continue
        raw_status = str(meta.status).strip().lower() if meta.status else "expired"

        transaction_start_date = clean_date(meta.transaction_start_date)
        closing_date = clean_date(meta.closing_date)
        date_of_transaction = clean_date(meta.date_of_transaction)
        transfer_date = closing_date or date_of_transaction or transaction_start_date

        for p in participants:
            if p.participant_type != "Client":
                continue
            client_id = client_email_to_id.get(p.participant_email)
            if client_id is None:
                continue
            role = p.participant_role
            if role in seller_roles:
                if raw_status in completed_sale_statuses:
                    ownership_end_date = transfer_date
                    is_current_owner = False
                else:
                    ownership_end_date = None
                    is_current_owner = True

                record_key = (
                    prop_id,
                    client_id,
                    transaction_start_date,
                    ownership_end_date,
                    is_current_owner
                )

                if record_key not in seen_ownership_records:
                    seen_ownership_records.add(record_key)
                    owner_rows.append((
                        property_owner_id,
                        prop_id,
                        client_id,
                        transaction_start_date,
                        ownership_end_date,
                        is_current_owner
                    ))
                    property_owner_id += 1
            elif role in buyer_roles and raw_status in completed_sale_statuses:
                record_key = (
                    prop_id,
                    client_id,
                    transfer_date,
                    None,
                    True
                )
                if record_key not in seen_ownership_records:
                    seen_ownership_records.add(record_key)
                    owner_rows.append((
                        property_owner_id,
                        prop_id,
                        client_id,
                        transfer_date,
                        None,
                        True
                    ))
                    property_owner_id += 1

    with conn.cursor() as cur:
        if owner_rows:
            execute_values(cur, """
                INSERT INTO PropertyOwners
                    (property_owner_id, property_id, client_id,
                     ownership_start_date, ownership_end_date, is_current_owner)
                VALUES %s
                ON CONFLICT DO NOTHING
            """, owner_rows)

    conn.commit()

#### **Insert Data into SQL Tables**

In [44]:
# validation

val = [
    ("Offices",                 "SELECT COUNT(*) FROM Offices"),
    ("Neighbourhoods",          "SELECT COUNT(*) FROM Neighbourhoods"),
    ("Clients",                 "SELECT COUNT(*) FROM Clients"),
    ("Agents",                  "SELECT COUNT(*) FROM Agents"),
    ("Properties",              "SELECT COUNT(*) FROM Properties"),
    ("PropertyOwners",          "SELECT COUNT(*) FROM PropertyOwners"),
    ("Appointments",            "SELECT COUNT(*) FROM Appointments"),
    ("AppointmentInteractions", "SELECT COUNT(*) FROM AppointmentInteractions"),
    ("OpenHouses",              "SELECT COUNT(*) FROM OpenHouses"),
    ("Listings",                "SELECT COUNT(*) FROM Listings"),
    ("Transactions",            "SELECT COUNT(*) FROM Transactions"),
    ("TransactionAgents",       "SELECT COUNT(*) FROM TransactionAgents"),
    ("TransactionParties",      "SELECT COUNT(*) FROM TransactionParties"),
    ("Commissions",             "SELECT COUNT(*) FROM Commissions"),
    ("Expenses",                "SELECT COUNT(*) FROM Expenses"),
]


def validate(conn):
    with conn.cursor() as cur:
        for label, sql in val:
            cur.execute(sql)
            count = cur.fetchone()[0]

In [45]:
# insert

def main():
    conn = connect()

    df_offices = load_csv("raw_offices.csv")
    df_clients = load_csv("raw_clients.csv")
    df_agents = load_csv("raw_agents.csv")
    df_properties = load_csv("raw_properties.csv")
    df_appts = load_csv("raw_appointments.csv")
    df_oh = load_csv("raw_openhouses.csv")
    df_tx = load_csv("raw_transactions.csv")

    office_name_to_id = load_offices(conn, df_offices)
    neigh_name_to_id = load_neighbourhoods(conn, df_properties)
    client_email_to_id = load_clients(conn, df_clients)
    agent_email_to_id = load_agents(conn, df_agents, office_name_to_id)
    prop_addr_to_id = load_properties(conn, df_properties, neigh_name_to_id)

    load_appointments(conn, df_appts, office_name_to_id, prop_addr_to_id, agent_email_to_id, client_email_to_id)
    load_openhouses(conn, df_oh, agent_email_to_id, prop_addr_to_id)
    load_transactions(conn, df_tx, prop_addr_to_id, agent_email_to_id, client_email_to_id)
    update_property_status(conn)
    load_property_owners(conn, df_tx, prop_addr_to_id, client_email_to_id)

    validate(conn)
    
    conn.close()

if __name__ == "__main__":
    main()